[<< SL-14 AIFeynman](SL-14-AIFeynman-Discover-Equations.ipynb) | [↑ Symbolic Learning](../README.md) | [SL-9 LLM + Oracle >>](SL-9-LLM-SymbolicLearning.ipynb)

# SL-15 — Conjectures apprises pour un vérificateur symbolique

**Neural diving lu en doctrine Symbolic Learning.**

Dans SL-9, un LLM proposait des règles et un oracle symbolique les validait ou les rejetait : *le générateur varie, l'oracle ne varie pas*. Ce notebook transpose la même grammaire à un terrain où l'oracle est un **solveur d'optimisation réel** (CP-SAT) et où l'apprenant propose, non des règles, mais des **conjectures sur la solution** : « telle variable vaut telle valeur ».

La question pédagogique n'est plus « le plongeon aide-t-il en moyenne ? » (c'est celle du benchmark App-33, série Search) mais **« laquelle de ces conjectures aide réellement le vérificateur, laquelle l'égare, et pourquoi »** — la lecture mécaniste, décomposée conjecture par conjecture.

| | SL-9 | SL-15 |
|---|---|---|
| Générateur | LLM → règles Horn | MLP → fixations de variables |
| Oracle | vérification symbolique des règles | CP-SAT : statut + compteur de branches |
| Question | la règle est-elle consistante ? | la conjecture accélère-t-elle ou égare-t-elle la preuve ? |
| Boucle | règle rejetée → relance | conjecture réfutée → retrait, re-solve |

Le problème d'application : la **coloration de graphe** — chaque sommet reçoit une couleur, deux sommets adjacents diffèrent. Un solveur CP-SAT cherche en alternant propagation et décisions ; si on lui *souffle* des valeurs, tantôt il gagne du temps, tantôt il s'égare. Ce notebook mesure et explique les deux.

**Prérequis** : SL-9 (boucle générateur/oracle). **Navlink croisé** : App-33 (série Search) porte le benchmark agrégé ; ce notebook-ci porte la mécanique et la doctrine.

## 1. Le cadre : conjecture, oracle, réfutation

Une **conjecture** est une fixation partielle : un sous-ensemble de variables avec une valeur proposée. L'**oracle** (CP-SAT) l'accepte comme *hint* (il reste libre de l'ignorer) et rend un verdict mesurable : le nombre de **branches** explorées.

La mesure de l'aide est un **différentiel** : `Δbranches = branches(hint) − branches(pur)`. Négatif = la conjecture a accéléré ; positif = elle a égaré ; nul = neutre.

On commence par l'oracle seul.

In [1]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
from ortools.sat.python import cp_model

NV, KMAX, DEG = 60, 6, 3          # 60 sommets, 6 couleurs max, ~3 aretes / sommet
SEED0, N_TRAIN, N_TEST, N_DECOMP = 5000, 60, 12, 5


def make_graph(seed: int) -> np.ndarray:
    """Graphe 60 sommets, DEG aretes / sommet, tirage seede par sommet."""
    r = np.random.default_rng(seed)
    adj = np.zeros((NV, NV), dtype=np.int8)
    for v in range(NV):
        for _ in range(DEG):
            u = int(r.integers(NV))
            if u != v:
                adj[v, u] = adj[u, v] = 1
    return adj


print("Imports charges : numpy + OR-Tools CP-SAT")

Imports charges : numpy + OR-Tools CP-SAT


### L'oracle

`solve_coloring` construit le modele CP-SAT de la coloration propre, minimise le nombre de couleurs, et — quand on lui passe un `hint` — l'offre comme point de depart. Il rend le **statut**, les **branches** (l'effort de preuve) et la solution. C'est notre oracle : déterministe (`random_seed=7`, un seul worker), il ne varie pas d'un appel à l'autre.

In [2]:
def solve_coloring(adj: np.ndarray, hint: np.ndarray | None = None,
                   time_limit: float = 15.0) -> tuple[int, int, float, np.ndarray | None]:
    """Resout la coloration propre de `adj` ; rend (couleurs, branches, temps, solution)."""
    model = cp_model.CpModel()
    c = [[model.NewBoolVar(f"c{v}_{j}") for j in range(KMAX)] for v in range(NV)]
    for v in range(NV):
        model.AddExactlyOne(c[v])
    for v in range(NV):
        for u in range(v + 1, NV):
            if adj[v, u]:
                for j in range(KMAX):
                    model.Add(c[v][j] + c[u][j] <= 1)
    used = [model.NewBoolVar(f"u{j}") for j in range(KMAX)]
    for j in range(KMAX):
        for v in range(NV):
            model.Add(used[j] >= c[v][j])
    model.Minimize(sum(used))
    if hint is not None:
        for v in range(NV):
            for j in range(KMAX):
                model.AddHint(c[v][j], int(hint[v, j]))
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.random_seed = 7
    solver.parameters.num_workers = 1
    t0 = time.perf_counter()
    status = solver.Solve(model)
    wall = time.perf_counter() - t0
    if status != cp_model.OPTIMAL:
        return 0, solver.NumBranches(), wall, None
    sol = np.array([[solver.Value(c[v][j]) for j in range(KMAX)] for v in range(NV)], dtype=np.int8)
    colors = int(sol.sum(axis=0).clip(0, 1).sum())
    return colors, solver.NumBranches(), wall, sol


adj0 = make_graph(SEED0)
k0, nodes0, wall0, sol0 = solve_coloring(adj0)
print(f"controle moteur : {k0} couleurs, {nodes0} branches, {wall0:.3f} s")

controle moteur : 4 couleurs, 2738 branches, 0.093 s


### La conjecture


Un apprenant de SL-15 propose des fixations sur la **solution**. Trois formes seront comparées :

| Conjecture | Forme | Question |
|---|---|---|
| **Individuelle** | une fixation `{v : j}` seule | accélère-t-elle la preuve à elle seule ? |
| **Conjointe top-k** | les `k` fixations les plus confiantes | l'union des meilleures conjectures est-elle meilleure que chacune ? |
| **Projetée** | le hint complet, réparé pour ne plus créer de conflit | la réparation préserve-t-elle le gain ? |

Avant tout apprenant, une contrainte structurelle : les noms de couleurs sont interchangeables. Deux solutions qui ne diffèrent que par une permutation des couleurs sont **la même** coloration — on les canonise par ordre de première occurrence.

In [3]:
def canonize(sol: np.ndarray) -> np.ndarray:
    """Reordonne les couleurs par ordre de premiere occurrence dans les sommets."""
    order: list[int] = []
    sol_c = sol.copy()
    for v in range(NV):
        j0 = int(np.argmax(sol[v]))
        if j0 not in order:
            order.append(j0)
    remap = {old: new for new, old in enumerate(order)}
    for v in range(NV):
        j0 = remap[int(np.argmax(sol[v]))]
        sol_c[v] = 0
        sol_c[v, j0] = 1
    return sol_c


rng = np.random.default_rng(123)
perm = rng.permutation(KMAX)
sol_perm = np.zeros_like(sol0)
for v in range(NV):
    sol_perm[v, int(perm[int(np.argmax(sol0[v]))])] = 1
aligned = (canonize(sol0) == canonize(sol_perm)).all()
print(f"canonize aligne une solution et sa permutation de couleurs : {aligned}")

canonize aligne une solution et sa permutation de couleurs : True


## 2. L'apprenant : imiter les solutions sœurs

L'apprenant est un MLP sklearn entraîné à prédire, sommet par sommet, la couleur canonisée d'instances **sœurs** (mêmes paramètres, graines différentes). Il ne voit jamais une instance de test avant la mesure : c'est la même discipline que la généralisation d'App-28 (Learning to Branch).

La sortie du MLP en forme one-hot *est* la conjecture conjointe « toutes les fixations d'un coup ».

In [4]:
Xtr, Ytr, ks_tr, nodes_tr = [], [], [], []
for s in range(SEED0, SEED0 + N_TRAIN):
    adj = make_graph(s)
    k, nd, _, sol = solve_coloring(adj)
    Xtr.append(adj.flatten())
    Ytr.append(canonize(sol).flatten())
    ks_tr.append(k)
    nodes_tr.append(nd)
print(f"corpus d'entrainement : {N_TRAIN} instances | couleurs med={np.median(ks_tr):.0f} "
      f"| branches med={np.median(nodes_tr):.0f}")

from sklearn.neural_network import MLPClassifier

Xtr = np.array(Xtr, dtype=float)
Ytr = np.array(Ytr, dtype=int)
mlp = MLPClassifier(hidden_layer_sizes=(128, 128), max_iter=3000, random_state=0)
mlp.fit(Xtr, Ytr)
print("MLP entraine sur les solutions soeurs canonisees")

corpus d'entrainement : 60 instances | couleurs med=4 | branches med=2742


MLP entraine sur les solutions soeurs canonisees


Un premier regard sur la conjecture conjointe, sur les instances de test : le hint complet (toutes les fixations prédites) comparé au solve pur. C'est le résultat agrégé — App-33 le porte pour 25 instances ; ici on le garde comme point de départ du mécanisme.

In [5]:
rows = []
for s in range(SEED0 + 1000, SEED0 + 1000 + N_TEST):
    adj = make_graph(s)
    p1 = mlp.predict_proba(adj.flatten().reshape(1, -1)).reshape(NV, KMAX)
    pred_oh = np.zeros((NV, KMAX), dtype=np.int8)
    pred_oh[np.arange(NV), np.argmax(p1, axis=1)] = 1
    kA, nA, tA, _ = solve_coloring(adj)
    kB, nB, tB, _ = solve_coloring(adj, hint=pred_oh)
    conflicts = sum(1 for v in range(NV) for u in range(v + 1, NV)
                    if adj[v, u] and int(np.argmax(pred_oh[v])) == int(np.argmax(pred_oh[u])))
    rows.append(dict(seed=s, nA=nA, nB=nB, tA=round(tA, 3), tB=round(tB, 3),
                     conflits=conflicts))
medA = float(np.median([r["nA"] for r in rows]))
medB = float(np.median([r["nB"] for r in rows]))
medC = float(np.median([r["conflits"] for r in rows]))
print(f"conjointe : medianes branches pur={medA:.0f} vs hint complet={medB:.0f} | conflits med={medC:.0f}")
delta_pct = 100.0 * (medB - medA) / medA
print(f"delta median = {delta_pct:+.1f} %")

conjointe : medianes branches pur=2938 vs hint complet=2396 | conflits med=42
delta median = -18.4 %


### Lecture du résultat

Sur ce run, le hint complet **réduit** la médiane de 2938 à 2396 branches (**−18,4 %**) — et il le fait tout en forçant une médiane de **42 arêtes conflictuelles** (~47 % des ~90 arêtes prédites). Un conseil majoritairement incompatible avec la contrainte reste donc profitable : la « propreté » du conseil n'est pas la condition de son utilité.

C'est déjà une leçon de doctrine, mais elle n'est pas celle qu'on attendait : *une conjecture n'a pas à être un début de solution valide pour accélérer la preuve*. Reste à savoir **quelles** fixations portent ce gain — la section suivante décompose le hint sommet par sommet.


## 3. La conjecture individuelle — décomposer le hint

La question mécaniste : si on ne souffle qu'**une seule** fixation à l'oracle, gagne-t-on des branches ? Pour chaque sommet `v`, on re-solve l'instance avec le hint réduit à `{v : argmax p1[v]}`, et on mesure `Δv = branches(hint_v) − branches(pur)`.

Le tri de ces `Δ` sépare trois populations : conjectures **aidantes** (Δ<0), **neutres** (Δ=0) et **égarantes** (Δ>0). C'est cette décomposition que le benchmark agrégé ne peut pas voir.

In [6]:
def individual_deltas(adj: np.ndarray, p1: np.ndarray, time_limit: float = 5.0):
    """Pour chaque sommet, solve avec la fixation seule {v: argmax} ; rend (delta, base, fixations)."""
    _, nA, _, _ = solve_coloring(adj, time_limit=time_limit)
    deltas, colors = [], []
    for v in range(NV):
        j = int(np.argmax(p1[v]))
        hint = np.zeros((NV, KMAX), dtype=np.int8)
        hint[v, j] = 1
        _, nv, _, _ = solve_coloring(adj, hint=hint, time_limit=time_limit)
        deltas.append(nv - nA)
        colors.append(j)
    return np.array(deltas, dtype=int), nA, colors


decomp_seeds = list(range(SEED0 + 1000, SEED0 + 1000 + N_DECOMP))
decomp = []
for s in decomp_seeds:
    adj = make_graph(s)
    p1 = mlp.predict_proba(adj.flatten().reshape(1, -1)).reshape(NV, KMAX)
    d, base, colors = individual_deltas(adj, p1)
    decomp.append(dict(seed=s, base=base, d=d, p1=p1))

all_d = np.concatenate([r["d"] for r in decomp])
aidantes = int((all_d < 0).sum())
neutres = int((all_d == 0).sum())
egarantes = int((all_d > 0).sum())
print(f"conjectures individuelles sur {N_DECOMP} instances ({all_d.size} solves) :")
print(f"  aidantes  (delta<0) : {aidantes:4d}  ({100*aidantes/all_d.size:.1f} %)")
print(f"  neutres   (delta=0) : {neutres:4d}  ({100*neutres/all_d.size:.1f} %)")
print(f"  egarantes (delta>0) : {egarantes:4d}  ({100*egarantes/all_d.size:.1f} %)")
print(f"  meilleur gain       : {all_d.min():+d} branches | pire cout : {all_d.max():+d}")
print(f"  mediane des deltas  : {np.median(all_d):+.1f}")

conjectures individuelles sur 5 instances (300 solves) :
  aidantes  (delta<0) :  300  (100.0 %)
  neutres   (delta=0) :    0  (0.0 %)
  egarantes (delta>0) :    0  (0.0 %)
  meilleur gain       : -1994 branches | pire cout : -422
  mediane des deltas  : -1865.0


### Lecture du résultat

Sur ce run, **chaque** conjecture individuelle aide : les 300 fixations testées (5 instances × 60 sommets) réduisent **toutes** le nombre de branches — de 422 à 1994 sur une base d'environ 2900 par instance, médiane **−1865** (≈ −63 %). Il n'y a ni population neutre ni queue égarante.

C'est le résultat le plus fin de la décomposition : **une seule fixation bien choisie vaut mieux que les soixante**. Le hint complet de la section 2 n'affichait que −18 % ; la conjecture conjointe perd donc les deux tiers du gain qu'une conjecture isolée procure. Les 42 arêtes conflictuelles de la conjointe forcent le solveur à réparer avant de progresser : **l'interférence entre conjectures — pas leur neutralité — érode le signal**. Empiler les conjectures est un choix, pas une nécessité : la section 4 cherche la taille de tête qui préserve le gain.

> Mesure déterministe (`num_workers=1`, `random_seed=7`, budget 5 s jamais atteint) : le détour d'une conjecture isolée se rejoue à l'identique.


## 4. Les conjectures de tête : top-k

Le tri se fait par **confiance** du MLP (probabilité de la couleur prédite) — l'apprenant ne connaît pas les `Δ`, il faut donc un critère qu'il *peut* calculer. On compare trois tailles de tête : k=10, k=20, et complet (60).

> **Exercice 1** (cellule suivante) : implémenter `hint_top_k` — ne fixer que les `k` sommets de plus grande confiance, laisser les autres libres (vecteur nul, pas de `AddHint`). Puis re-mesurer la médiane `Δbranches` pour k=10 et k=20.

In [7]:
def hint_top_k(p1: np.ndarray, k: int | None = None) -> np.ndarray:
    # TODO etudiant : ne fixer que les k sommets les plus confiants (0 partout ailleurs).
    # Indice : conf = p1[i, argmax(p1[i])] ; trier les indices par confiance decroissante ;
    #          ne poser la fixation que pour les k premiers.
    # Version de repli : hint complet (comportement mesure en section 2).
    pred = np.zeros_like(p1, dtype=np.int8)
    pred[np.arange(NV), np.argmax(p1, axis=1)] = 1
    return pred


print("Exercice a completer : hint_top_k (repli = hint complet, la section 2 se rejoue)")

Exercice a completer : hint_top_k (repli = hint complet, la section 2 se rejoue)


## 5. La conjecture projetée — l'oracle répare l'apprenant

Repérer les conflits est une opération d'oracle, pas d'apprenant : deux fixations `{v : j}` et `{u : j}` sur une arête `(v,u)` sont **contradictoires** avec toute coloration propre. La boucle de doctrine SL : *l'apprenant propose, l'oracle réfute, on répare, on re-soumet*.

> **Exercice 2** : implémenter `oracle_check_conjectures` — détecter la liste des arêtes en conflit dans une conjecture conjointe.

In [8]:
def oracle_check_conjectures(pred_oh: np.ndarray, adj: np.ndarray) -> list[tuple[int, int]]:
    # TODO etudiant : rendre la liste des aretes (v, u) de adj ou les deux extremites
    # portent la meme couleur fixee. Indice : couleur de v = argmax(pred_oh[v]).
    # Version de repli : aucune arete en conflit (liste vide).
    return []


conflits0 = oracle_check_conjectures(
    np.eye(KMAX, dtype=np.int8)[np.argmax(
        mlp.predict_proba(make_graph(decomp_seeds[0]).flatten().reshape(1, -1)).reshape(NV, KMAX), axis=1)],
    make_graph(decomp_seeds[0]))
print(f"controle : conjectures en conflit sur l'instance {decomp_seeds[0]} : {len(conflits0)}")

controle : conjectures en conflit sur l'instance 6000 : 0


> **Exercice 3** : implémenter `project_faisable` — recoller les fixations en conflit *gloutonnement* : pour chaque arête en conflit, **retirer** une des deux fixations (la moins confiante), jusqu'à zéro conflit. C'est la réparation minimale : l'oracle corrige l'apprenant sans changer de couleur ailleurs.

In [9]:
def project_faisable(pred_oh: np.ndarray, adj: np.ndarray) -> np.ndarray:
    # TODO etudiant : retirer gloutonnement une fixation par arete en conflit (la moins
    # confiante), jusqu'a 0 conflit. Indice : reutiliser oracle_check_conjectures.
    # Version de repli : hint inchange (conflits conserves, comportement mesure).
    return pred_oh.copy()


print("Exercice a completer : project_faisable (repli = hint inchange)")

Exercice a completer : project_faisable (repli = hint inchange)


## 6. Synthèse : la conjecture comme hypothèse réfutable

| Constat mesuré (run courant) | Lecture de doctrine |
|---|---|
| Hint complet : médiane **2938 → 2396** branches (−18,4 %), **42 conflits médians** (~47 % des arêtes) | un conseil majoritairement conflictuel produit quand même un gain net — la précision n'est pas la condition de l'utilité (cohérent avec App-33 : 25/25 instances améliorées, run déterministe) |
| Décomposition : **300/300 conjectures individuelles aidantes**, médiane **−1865** (≈ −63 %) | **une** fixation vaut mieux que soixante : le signal s'érode par **interférence**, pas par dilution de conjectures neutres |
| L'oracle détecte et répare (conflits listables, réparation gloutonne) | la boucle générateur/oracle de SL-9 fonctionne sur des fixations comme sur des règles |
| Le solveur reste déterministe et **optimal** avec ou sans hint | l'oracle ne varie pas : la conjecture est l'hypothèse, la preuve est le verdict |

**Ce que ce notebook a ajouté à la série** : la boucle conjecture → réfutation → réparation, sur un oracle d'optimisation réel. SL-9 portait la boucle sur des règles logiques (restaurant AIMA) ; SL-15 la porte sur des instances combinatoires et un solveur SOTA (CP-SAT) — le même geste neuro-symbolique, mesuré en branches de preuve.

**Verdict honnête de l'étude** (famille coloration 60 sommets, DEG=3, run déterministe) : la conjecture apprise **aide** — massivement à l'unité (−63 % médian), modérément en bloc (−18 %) — et le plongeon paie même conflictuel. Le résultat négatif qui structure la doctrine n'est donc pas « le plongeon dégrade » (une lecture antérieure du prototype, artefact de parallélisme documenté par App-33) mais **« l'empilement dilue »** : le signal vit dans une conjecture isolée bien choisie, et le rôle de l'oracle est de réfuter et réparer le reste. C'est exactement la morale neuro-symbolique de la série : l'apprenant propose, le vérificateur dispose.


**Bibliographie** : Nair et al. 2021, *Solving Mixed Integer Programs Using Neural Networks* — au gisement partagé (`Bibliographie IA/Search/`). Le benchmark agrégé et les familles d'instances : App-33 (série Search). La frontière anti-duplication et le suivi : issue #17605.